# This was a dead end! The number of degrees of freedom is equal to constraints with reduced cost=0

You can't easily find the size of the solution space, because you have degenerate variables. 

**Conclusion**: The best solution is to sample the solution space and find the variability in the solutions.



In [1]:
import reframed
import cobra
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

In [2]:
from reframed.core.model import ReactionType
from reframed.solvers import solver_instance
from math import inf, isinf
from reframed.community.solution import CommunitySolution
from reframed.solvers.solution import Status
from numpy.random import lognormal
from reframed.core.elements import molecular_weight

In [3]:
import argparse
import textwrap
from reframed.community.model import Community
from reframed.solvers.solution import Status

import glob
import pandas as pd
from reframed.io.cache import ModelCache
import os
from reframed import Environment, set_default_solver
from math import inf

In [4]:
import steadiercom.steadiercom as steadiercom
from steadiercom.steadiercom import allocation_constraints
from steadiercom.cli import load_communities,load_media_db
from steadiercom.steadiercom import build_problem 

In [5]:
models = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/species_used/"):
        models[path[:-4]] = reframed.load_cbmodel("../models/species_used/"+path,exchange_detection="R_EX_(?!biomass__40__e__41__)")
    

In [6]:
def SteadierSample(community, objective=None, growth=0.1, abundance=None, allocation=False, constraints=None,
             w_e=0.002, w_r=0.2, solver=None):

    if abundance:
        norm = sum(abundance.values())
        abundance = {org_id: abundance.get(org_id, 0) / norm for org_id in community.organisms}

    if not solver:
        solver = build_problem(community, growth=growth, abundance=abundance)

    if not allocation:
        w_e, w_r = 0, 0

    random_vars = allocation_constraints(community, solver, w_e=w_e, w_r=w_r, abundance=abundance)

    sols = []
    if abundance:
        w1 = {org_id: 1 / abundance[org_id] if abundance[org_id] > 0 else 0 for org_id in community.organisms}


    #if growth is None and objective is None:
    objective = community.merged_model.biomass_reaction

    #if objective is not None:
    #    sol = solver.solve(objective, minimize=False, constraints=constraints)
    #    if constraints is None:
    #        constraints = {objective: (sol.fobj, inf)} 
    #    else:
    #        constraints[objective] = (sol.fobj, inf)

    
    status = Status.INFEASIBLE
    i = 0
    while status == Status.INFEASIBLE:
        
        print(f"iteration {i}")
        #objective = {vi: w1[org_id] * lognormal(0, 1) for org_id, v_org in random_vars.items() for vi in v_org}

        sol = solver.solve(objective, minimize=True, constraints=constraints,shadow_prices=True, reduced_costs=True)
        i +=1
        status = sol.status
    return sol,solver

In [7]:
mediadb = "../data/Venturelli2018/generated/media_db_lb_extend.tsv"#path to mediadb
media = ["LB_extend"]
growth = 0.039683# growth rate
target = None

models = ["../models/species_used/"+name for name in listdir("../models/species_used/")]


communities = "../data/Venturelli2018/generated/communities_12_exp_abundance.tsv"

comm_id = "NONE"

In [8]:
#def main_run(models, communities=None, output=None, media=None, mediadb=None, growth=None, sample=None, 
#             w_e=None, w_r=None, target=None, unlimited=None):
w_e=None
w_r=None
abstol = 1e-9
model_cache, comm_dict, has_abundance = load_communities(models, communities, exchange_detection='R_EX_(?!biomass__40__e__41__)')

media_db, media_has_bounds = load_media_db(mediadb)
results = []

organisms = comm_dict[comm_id]

abundance = organisms
organisms = organisms.keys()
comm_models = [model_cache.get_model(org_id, reset_id=True) for org_id in organisms]
community = Community(comm_id, comm_models, copy_models=False)


for medium in media:
    #env = Environment.from_compounds(media_db[medium],max_uptake=10).apply(community.merged_model, inplace=False, exclusive=True, warning=False)
    
    #env = Environment.complete(community.merged_model).apply(community.merged_model, inplace=False, exclusive=True, warning=False)

    env = Environment.from_compounds(media_db[medium],max_uptake=10)
    env.apply(community.merged_model, inplace=False, exclusive=True, warning=False)
    env_dict = dict(env)

    if media_has_bounds:
        for cpd, bound in media_db[medium].items():
            r_id = f'R_EX_{cpd}_e'
            if r_id in env:
                env[r_id] = (-bound, inf)

    print(f'simulating {comm_id} in {medium} medium')


    sol,solver = SteadierSample(community, abundance=abundance, growth=growth, allocation=True, constraints=env, w_e=w_e, w_r=w_r, objective=target)

/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/reframed/io/sbml.py:574: UserWarning: Could not extract annotation from http://identifiers.org/inchi/InChI=1S/C12H21N3O13P2/c13-8-1-2-15(12(20)14-8)11-10(19)9(18)7(27-11)5-26-30(23,24)28-29(21,22)25-4-6(17)3-16/h1-2,6-7,9-11,16-19H,3-5H2,(H,21,22)(H,23,24)(H2,13,14,20)/
  warn(f'Could not extract annotation from {term.getResourceURI(i)}')
/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/reframed/io/sbml.py:574: UserWarning: Could not extract annotation from http://identifiers.org/inchi/InChI=1S/C11H12Cl2N2O5/c12-10(13)11(18)14-8(5-16)9(17)6-1-3-7(4-2-6)15(19)20/h1-4,8-10,16-17H,5H2,(H,14,18)/
  warn(f'Could not extract annotation from {term.getResourceURI(i)}')
/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/reframed/io/sbml.py:574: UserWarning: Could not extract annotation from http://identifiers.org/inchi/InChI=1S/C15H26N2O12/c1-4(20)

simulating NONE in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-14
iteration 0


/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/reframed/solvers/gurobi_solver.py:143: UserWarning: Constrained variable 'R_EX_R_EX_ac__91__e__93___e' not previously declared
  warn(f"Constrained variable '{r_id}' not previously declared")
/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/reframed/solvers/gurobi_solver.py:143: UserWarning: Constrained variable 'R_EX_R_EX_ribflv__91__e__93___e' not previously declared
  warn(f"Constrained variable '{r_id}' not previously declared")
/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/reframed/solvers/gurobi_solver.py:143: UserWarning: Constrained variable 'R_EX_R_EX_asn_L__91__e__93___e' not previously declared
  warn(f"Constrained variable '{r_id}' not previously declared")
/Users/idunmariaburgos/opt/anaconda3/envs/steadiercom/lib/python3.10/site-packages/reframed/solvers/gurobi_solver.py:143: UserWarning: Constrained variable 'R_EX_R_EX_f

In [9]:
sol

Objective: 0.039683
Status: Optimal

In [10]:
len({met for met,value in sol.shadow_prices.items() if abs(value)<1e-6})

52880

In [11]:
len(sol.shadow_prices)

52892

In [12]:
set(sol.shadow_prices.keys())-set({met for met,value in sol.shadow_prices.items() if abs(value)<1e-6})

{'community_biomass',
 'g_Bacteroides_ovatus_ATCC_8483',
 'g_Bacteroides_thetaiotaomicron_VPI_5482',
 'g_Bacteroides_uniformis_ATCC_8492',
 'g_Bacteroides_vulgatus_ATCC_8482',
 'g_Blautia_hydrogenotrophica_DSM_10507',
 'g_Clostridium_hiranonis_TO_931_DSM_13275',
 'g_Collinsella_aerofaciens_ATCC_25986',
 'g_Desulfovibrio_piger_ATCC_29098',
 'g_Eggerthella_lenta_DSM_2243',
 'g_Eubacterium_rectale_ATCC_33656',
 'g_Faecalibacterium_prausnitzii_A2_165'}